<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/ToeknGenerationFromModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This program will show a sample on how different model would generate the tokens
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer
import gradio as gr
from functools import lru_cache

# -----------------------------
# 🔐 Login to Hugging Face
# -----------------------------
hf_token = userdata.get('HF_TOKEN')

if hf_token and hf_token.startswith("hf_"):
    print("✅ HF key looks good")
    login(hf_token)
else:
    print("❌ HF key missing")


# -----------------------------
# ⚡ Lazy Load Tokenizers (BEST PRACTICE)
# -----------------------------
MODEL_MAP = {
    "MiniMaxAI": "MiniMaxAI/MiniMax-M2.7",
    "zai-org": "zai-org/GLM-5.1",
    "datalab-to": "datalab-to/chandra-ocr-2",
    "LiquidAI": "LiquidAI/LFM2.5-VL-450M"
}


@lru_cache(maxsize=4)
def get_tokenizer(model_name):
    print(f"🚀 Loading tokenizer for {model_name} (only once)")

    model_path = MODEL_MAP[model_name]

    return AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True
    )


# -----------------------------
# 🧠 Main Function
# -----------------------------
def type_of_model_tokens(model_name, text):
    if not model_name:
        return "⚠️ Please select a model"

    if not text:
        return "⚠️ Please enter text"

    tokenizer = get_tokenizer(model_name)

    tokens = tokenizer.tokenize(text)

    # Optional: show token count also
    return f"Tokens ({len(tokens)}):\n{tokens}"


# -----------------------------
# 🎨 Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 Multi-Model Tokenizer Explorer")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(MODEL_MAP.keys()),
            label="Select Model"
        )

        text_input = gr.Textbox(
            lines=5,
            label="Enter Text"
        )

    output = gr.Textbox(
        lines=10,
        label="Tokenized Output"
    )

    run_btn = gr.Button("Tokenize")

    run_btn.click(
        fn=type_of_model_tokens,
        inputs=[model_dropdown, text_input],
        outputs=output
    )


# -----------------------------
# 🚀 Launch
# -----------------------------
demo.launch(debug=True)